This Lab is Data Preprocessing Project

## Lab sentiment analysis (NLP)


# Table of Contents 
<ol start="1">
<li> About the project</li>
<li> Loading and Cleaning with Pandas</li>
<li> Data cleaning </li>
<li> Feature engineering </li>
<li> Text Preprocessing </li>
<li> build Model </li>
<li> Visualization </li>
<li> Conclusion </li>
</ol>

## Problem Statement
The surge in MOOC learner reviews demands efficient data preprocessing for subsequent model readiness. Manual analysis is impractical due to the data volume. This project focuses on crucial preprocessing steps—cleaning, feature engineering, normalization, and transformation—to optimize the data for sentiment analysis models. The goal is to enable MOOC platforms to derive actionable insights, improve course quality, and identify areas for enhancement through systematic and effective data preparation. 

## Data Source
Udemy Courses- Comments.csv- Kaggle: "This dataset contains detailed information on all available Udemy courses on Oct 10, 2022. This data was provided in the "Course_info.csv" file. Also, over 9 million comments were collected and provided in the "Comments.csv" file. The information of over 209k courses was collected by web scraping the Udemy website. Udemy holds 209,734 courses and 73,514 instructors teaching courses in 79 languages in 13 different categories." --Kaggle.
In this Project we only use Comments.csv.

import appropriate libraries

In [1]:
%pip install numpy pandas nltk scikit-learn tqdm beautifulsoup4 pyspark emot langdetect translators

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary libraries
import numpy as np   # NumPy for numerical operations
import pandas as pd  # Pandas for data manipulation
from nltk.tokenize import word_tokenize  # NLTK for natural language processing - tokenization
from nltk.corpus import stopwords  # NLTK for stop words
from nltk.stem.lancaster import LancasterStemmer  # NLTK for stemming
from nltk.stem.wordnet import WordNetLemmatizer  # NLTK for lemmatization
from sklearn.feature_extraction.text import TfidfVectorizer  # Scikit-learn for TF-IDF vectorization
from sklearn.model_selection import train_test_split  # Scikit-learn for train-test split
from tqdm import tqdm  # tqdm for progress bars
from bs4 import BeautifulSoup  # BeautifulSoup for HTML parsing
from pyspark.sql.functions import col

# Importing emot library for emotion analysis

import emot 

# Regular expression library for text processing
import re

# Language detection library

from langdetect import detect

# Translators library for language translation
import os

os.environ["translators_default_region"] = "EN"

import translators as ts


In [3]:
import pyspark; print(pyspark.__version__)

4.2.0


In [4]:
import os

# Tell HDFS to treat this Spark job as the 'hadoop' user
os.environ["HADOOP_USER_NAME"] = "hadoop"

In [5]:
from pyspark.sql import SparkSession

# Stop any existing ghost sessions just to be safe
try:
    spark.stop()
except:
    pass

# Force the connection to your Spark Master container
spark = SparkSession.builder \
    .appName("MOOC_Distributed_Processing") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()
# Verify the connection
print(spark.sparkContext.master)

spark://spark-master:7077


In [6]:
# Load dataset with multiline and quote options enabled
df_spark = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .csv("hdfs://namenode:9000/data/mooc/raw/1000_Comments.csv")

In [7]:
# 1. Print Data Types and Column Structure (like the data type column in info())
df_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- course_id: integer (nullable = true)
 |-- rate: double (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- display_name: string (nullable = true)
 |-- comment: string (nullable = true)



In [8]:
# 2. Compute Total Row Count and Non-Null Counts per Column
total_rows = df_spark.count()
print(f"Total Rows: {total_rows}\n")


Total Rows: 739



In [9]:
print("Column Name | Data Type | Null Count | Non-Null Count")
print("-" * 55)
for c in df_spark.dtypes:
    col_name, col_type = c[0], c[1]
    null_count = df_spark.filter(col(col_name).isNull()).count()
    non_null_count = total_rows - null_count
    print(f"{col_name} | {col_type} | {null_count} | {non_null_count}")

Column Name | Data Type | Null Count | Non-Null Count
-------------------------------------------------------
id | int | 0 | 739
course_id | int | 0 | 739
rate | double | 0 | 739
date | timestamp | 0 | 739
display_name | string | 0 | 739
comment | string | 0 | 739


In [10]:
# Compute basic statistics for numeric columns (like rate, length, etc.)
df_spark.describe().show()

+-------+--------------------+------------------+------------------+------------+----------+
|summary|                  id|         course_id|              rate|display_name|   comment|
+-------+--------------------+------------------+------------------+------------+----------+
|  count|                 739|               739|               739|         739|       739|
|   mean|1.0222949695534506E8|3924803.1447902573| 4.106224627875507|        NULL|      NULL|
| stddev|2.2571025901300754E7| 784169.6594403044|1.3524600524142607|        NULL|      NULL|
|    min|             4378632|            773174|               0.5|     4playin|         ,|
|    max|           126648502|           4913148|               5.0|        青山|音频听不清|
+-------+--------------------+------------------+------------------+------------+----------+



In [11]:
# Total rows vs unique rows
total_count = df_spark.count()
unique_count = df_spark.dropDuplicates().count()

dup_count = total_count - unique_count

print(f"There are {dup_count} duplicate values in the dataset")

There are 0 duplicate values in the dataset


In [12]:
# Check duplicates specifically by comment ID
dup_count_id = total_count - df_spark.dropDuplicates(["id"]).count()
print(f"There are {dup_count_id} duplicate comments based on ID")

There are 0 duplicate comments based on ID


In [13]:


# Drop rows with null comments or course IDs/rates
df_spark = df_spark.dropna(subset=["comment"])

# Drop duplicates based on comment 'id'
df_spark = df_spark.dropDuplicates(["id"])

# Drop unused columns (similar to your pandas drop step)
df_spark = df_spark.drop("id", "display_name", "course_id")

In [14]:
##TODO not needed remove it 
from pyspark.sql.functions import to_timestamp

# Convert date string to timestamp
df_spark = df_spark.withColumn("date", to_timestamp(col("date")))

In [15]:
from pyspark.sql.functions import length, trim, regexp_replace

# Remove non-word characters and digits, check if resulting string is empty
cleaned_comment_expr = trim(regexp_replace(regexp_replace(col("comment"), r"\W+", ""), r"\d+", ""))

# Filter out rows where the cleaned comment length is 0 (equivalent to your check)
df_spark = df_spark.filter(length(cleaned_comment_expr) > 0)

In [16]:
# 2. Handle Missing Values & Duplicates
# Drop rows where critical fields (comment or rate) are null, and remove duplicate comments based on 'id'
df_cleaned = df_spark.dropna(subset=["comment", "rate"]) 


In [17]:
# 3. Drop unnecessary or administrative columns (adjust based on your actual schema)
# For example, removing display names or redundant identifiers if not needed for analysis
columns_to_drop = ["display_name"]
existing_drop_cols = [c for c in columns_to_drop if c in df_cleaned.columns]
df_cleaned = df_cleaned.drop(*existing_drop_cols)

In [18]:
# # Create the 5 groups WITHOUT limiting them
df_0_1 = df_cleaned.filter((col("rate") >= 0) & (col("rate") < 1))
df_1_2 = df_cleaned.filter((col("rate") >= 1) & (col("rate") < 2))
df_2_3 = df_cleaned.filter((col("rate") >= 2) & (col("rate") < 3))
df_3_4 = df_cleaned.filter((col("rate") >= 3) & (col("rate") < 4))
df_4_5 = df_cleaned.filter((col("rate") >= 4) & (col("rate") <= 5))

# Find the size of each group
sizes = [
    df_0_1.count(),
    df_1_2.count(),
    df_2_3.count(),
    df_3_4.count(),
    df_4_5.count()
]

# Smallest group size
shape = min(sizes)

print("Group sizes:", sizes)
print("Number of rows per group:", shape)
# Limit every group to the smallest size
df_0_1 = df_0_1.limit(shape)
df_1_2 = df_1_2.limit(shape)
df_2_3 = df_2_3.limit(shape)
df_3_4 = df_3_4.limit(shape)
df_4_5 = df_4_5.limit(shape)

# Combine them
df_sampled = (
    df_0_1
    .union(df_1_2)
    .union(df_2_3)
    .union(df_3_4)
    .union(df_4_5)
)

Group sizes: [0, 80, 37, 63, 508]
Number of rows per group: 0


In [19]:
df_sampled.head()

Row(rate=1.0, date=datetime.datetime(2020, 3, 26, 22, 20, 31), comment='no cohesion very quick')

In [20]:
df_sampled.count()

187

In [21]:
# Feature Engineering: Add sentiment column based on rate
from pyspark.sql.functions import col, when
df_sampled = df_sampled.withColumn(
    "sentiment", when(col("rate") < 2, "negative").when((col("rate") >= 2) & (col("rate") <= 3), "neutral").otherwise("positive")
)

In [22]:
# df_sampled.show()

In [23]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
import emot
# Regular expression library for text processing
import re
from langdetect import detect
import translators as ts
from bs4 import BeautifulSoup
import nltk
nltk.data.path.insert(0, '/usr/local/nltk_data')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.wordnet import WordNetLemmatizer

# NLTK setup on worker nodes
nltk.download('stopwords')
nltk.download("punkt_tab")
nltk.download("punkt")
nltk.download('wordnet')

emot_obj = emot.core.emot()
stop_words = set(stopwords.words('english')) - {'very','too','so','not','no','but'}
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [24]:
# ---------------------------------------------------------
# 1. Language Detection
# ---------------------------------------------------------
def detect_lang(text):
    if not text: return 'undetermined'
    try:
        return detect(str(text))
    except:
        return 'undetermined'

detect_lang_udf = udf(detect_lang, StringType())
df_processed = df_sampled.withColumn("detected_language", detect_lang_udf(col("comment")))

# Count and print "undetermined" languages
undetermined_count = df_processed.filter(col("detected_language") == "undetermined").count()
print(f"Count of 'undetermined' language comments: {undetermined_count}")

/usr/local/spark/python/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


Count of 'undetermined' language comments: 0


In [25]:
# ---------------------------------------------------------
# 2. Translation
# ---------------------------------------------------------
def translate_comment(text, lang):
    if not text or lang == 'undetermined':
        return text
    if lang != 'en':
        try:
            return ts.translate_text(text)
        except:
            return "Error in translation"
    return text

translate_udf = udf(translate_comment, StringType())
df_processed = df_processed.withColumn("translated_comment", translate_udf(col("comment"), col("detected_language")))

# Count, print, and filter out translation failures
failed_translation_count = df_processed.filter(col("translated_comment") == "Error in translation").count()
print(f"Count of failed translations: {failed_translation_count}")

df_processed = df_processed.filter(col("translated_comment") != "Error in translation")

Count of failed translations: 67


In [26]:
# # ---------------------------------------------------------
# # 3. Text Cleaning Pipeline
# # ---------------------------------------------------------
# def process_text_pipeline(text):
#     if not text: return text
    
#     # Replace emojis
#     try:
#         emojis = emot_obj.emoji(text)
#         for emoji, meaning in zip(emojis['value'], emojis['mean']):
#             text = text.replace(emoji, re.sub(r'[^\w\s]', '', meaning))
#     except:
#         pass
        
#     # Clean HTML and links
#     text = BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)
#     text = re.sub(r'http[s]?://\S+', '', text)
    
#     # Remove numbers/special chars and lowercase
#     text = text.replace("n't", " not").replace("_", " ")
#     text = re.sub(r'\W+', ' ', text)
#     text = re.sub(r'\d+', ' ', text).lower()
    
#     # Tokenization & Lemmatization
#     try:
#         tokens = [lemmatizer.lemmatize(t) for t in word_tokenize(text) if t.isalpha() and t not in stop_words]
#         return ' '.join(tokens)
#     except:
#         return text

In [27]:
# ---------------------------------------------------------
# 3. Text Cleaning Pipeline
# ---------------------------------------------------------
def process_text_pipeline(text):
    if not text: return text
    
    # Replace emojis
    emojis = emot_obj.emoji(text)
    for emoji, meaning in zip(emojis['value'], emojis['mean']):
        text = text.replace(emoji, re.sub(r'[^\w\s]', '', meaning))
        
    # Clean HTML and links
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # Remove numbers/special chars and lowercase
    text = text.replace("n't", " not").replace("_", " ")
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\d+', ' ', text).lower()
    
    # Tokenization & Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in word_tokenize(text) if t.isalpha() and t not in stop_words]
    return ' '.join(tokens)

In [28]:
process_text_pipeline("i need my certificate 😄")

'need certificate grinning face smiling eye'

In [29]:
# Apply preprocessing UDF on the translated text
clean_udf = udf(process_text_pipeline, StringType())
df_processed = df_processed.withColumn("clean_text", clean_udf(col("translated_comment")))



In [33]:
df_processed.show()

+----+-------------------+--------------------+---------+-----------------+--------------------+--------------------+
|rate|               date|             comment|sentiment|detected_language|  translated_comment|          clean_text|
+----+-------------------+--------------------+---------+-----------------+--------------------+--------------------+
| 1.0|2020-03-26 22:20:31|no cohesion very ...| negative|               en|no cohesion very ...|no cohesion very ...|
| 1.0|2020-04-23 06:38:27|The instructor do...| negative|               en|The instructor do...|instructor not pr...|
| 1.0|2020-05-26 00:20:56|Terrible, awful, ...| negative|               en|Terrible, awful, ...|terrible awful wo...|
| 1.0|2020-07-16 23:44:36|This package has ...| negative|               en|This package has ...|package numerous ...|
| 1.0|2020-07-25 17:08:43|Reference image i...| negative|               en|Reference image i...|reference image n...|
| 1.0|2020-07-28 13:07:22|Too many wrong an...| negative

In [32]:
# Save processed data to HDFS as Parquet
df_processed.write.mode("overwrite").parquet("hdfs://namenode:9000/data/mooc/processed/Comments_cleaned.parquet")

# Conclusion
In conclusion , the sentiment analysis project successfully navigated the challenges posed by the vast volume of MOOC learner feedback. The combination of thorough data preprocessing and advanced NLP techniques paved the way for a robust sentiment analysis model. Through exploratory data analysis, key insights were gleaned, and visualizations provided a nuanced understanding of the dataset. The meticulous cleaning process, feature engineering, and adoption of the RoBERTa model contributed to the creation of an accurate and effective sentiment analysis tool. This project not only streamlined the evaluation of MOOCs but also demonstrated the potential of NLP in extracting valuable insights from unstructured textual data.